# SpaceX Falcon 9 – Web Scraping Wikipedia launch records

In [1]:
import requests, re, unicodedata, pandas as pd
from bs4 import BeautifulSoup
url = 'https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'
soup = BeautifulSoup(requests.get(url, headers={'User-Agent':'Mozilla/5.0 (capstone)'}).text, 'html.parser')
soup.title

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>

In [2]:
def extract_column_from_header(row):
    for t in row.find_all(['br','a','sup']): t.extract()
    name = ' '.join(row.contents)
    if not name.strip().isdigit(): return name.strip()
tables = soup.find_all('table', 'wikitable plainrowheaders collapsible')
column_names = [c for c in (extract_column_from_header(th) for th in tables[0].find_all('th')) if c]
column_names

['Flight No.',
 'Date and time ( )',
 'Launch site',
 'Payload',
 'Payload mass',
 'Orbit',
 'Customer',
 'Launch outcome']

In [3]:
def date_time(t): return [d.strip() for d in list(t.strings)][0:2]
def booster_version(t): return ''.join([v for i,v in enumerate(t.strings) if i%2==0][0:-1])
def landing_status(t): return [i for i in t.strings][0]
def get_mass(t):
    m = unicodedata.normalize('NFKD', t.text).strip()
    return m[0:m.find('kg')+2] if m else 0
launch_dict = {k: [] for k in ['Flight No.','Launch site','Payload','Payload mass','Orbit','Customer','Launch outcome','Version Booster','Booster landing','Date','Time']}
for table in tables:
    for rows in table.find_all('tr'):
        if not (rows.th and rows.th.string and rows.th.string.strip().isdigit()): continue
        row = rows.find_all('td')
        if len(row) < 9: continue
        launch_dict['Flight No.'].append(rows.th.string.strip())
        dt = date_time(row[0]); launch_dict['Date'].append(dt[0].strip(',')); launch_dict['Time'].append(dt[1] if len(dt)>1 else None)
        bv = booster_version(row[1]) or row[1].a.string; launch_dict['Version Booster'].append(bv)
        launch_dict['Launch site'].append(row[2].a.string if row[2].a else row[2].text.strip())
        launch_dict['Payload'].append(row[3].a.string if row[3].a else row[3].text.strip())
        launch_dict['Payload mass'].append(get_mass(row[4])); launch_dict['Orbit'].append(row[5].a.string if row[5].a else row[5].text.strip())
        launch_dict['Customer'].append(row[6].a.string if row[6].a else row[6].text.strip())
        launch_dict['Launch outcome'].append(list(row[7].strings)[0].strip()); launch_dict['Booster landing'].append(landing_status(row[8]).strip())
df = pd.DataFrame(launch_dict); df.to_csv('spacex_web_scraped.csv', index=False)
print(df.shape); df.head()

(121, 11)


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.1,No,1 March 2013,15:10
